# Get Fanwork Links
This notebook contains functions that:
1) Parses a list of pagination links for works associated with any given Ao3 fandom (pagination link example: 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3').
2) For each pagination link, scrapes that link for a list of links to fanworks (fanwork link example: 'https://archiveofourown.org/works/80602681/chapters/211688701'.
3) Compiles all fanwork links into a list.

In [112]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup
import re

# Get pagination links list from txt file

In [2]:
def get_links_to_scrape(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    links_to_scrape = open(filepath).readlines()
    return links_to_scrape

# Scrape individual pagination link for list of fanwork links

In [128]:
def get_html_doc_pagination(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()
    else:
        pass

    # return page source / html doc
    driver.implicitly_wait(10) # seconds

    # find /html/body/div[1]/div[1]/div/ol[2]
    body = driver.find_element(By.XPATH, '/html/body/div[1]/div[1]/div/ol[2]')
    body_outer = body.get_attribute('outerHTML')

    # get BeautifulSoup object
    soup = BeautifulSoup(body_outer, 'html.parser')

    return soup

In [4]:
def get_all_links(soup):
    """Takes in a BeautifulSoup object and returns a list of all links in the html document."""
    links = soup.find_all('a')
    for link in links:
        link = link.get('href')
    return links

In [20]:
def get_urls_only(links_list):
    """Takes in a list of all links from get_all_urls and returns a list of only the URL strings."""
    # use regex to get just URLs
    urls_only = []
    pattern = r'\"(.*?)\"'
    for i in range(len(links_list)):
        link_str = str(links_list[i])
        match = re.search(pattern, link_str)
        if match:
            urls_only.append(match.group(1))
        else:
            continue

    # update URLs if they do not include 'http://archiveofourown.org' at the start
    for i in range(len(urls_only)):
        if urls_only[i][0] == '/':
            urls_only[i] = 'https://archiveofourown.org' + urls_only[i]
        else:
            continue

    return urls_only

In [7]:
def get_work_links(urls_only):
    """Pares down a list of URLs from get_urls_only to a list of URLs associated with fanworks (as opposd to Ao3 tags, bookmarks, collections, search, users, etc."""
    work_links = []
    for i in range(len(urls_only)):
        if 'https://archiveofourown.org/works/' in urls_only[i]:
            work_links.append(urls_only[i])
        else:
            continue
    return work_links

In [8]:
def pare_down_work_links(work_links):
    """Takes a list of fanwork links from get_work_links and pares down to links that only end in digits (i.e. the link to the first page of an individual fanwork, rather than links to kudos, comments, chapters, etc. associated with that fanwork)."""
    works_only = []
    pattern = r'\d+$'
    for i in range(len(work_links)):
        match = re.search(pattern, work_links[i])
        if match:
            works_only.append(work_links[i])
        else:
            continue

    return works_only

In [66]:
def remove_chapter_links(work_links_pared):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only = work_links_pared
    for link in works_only:
        if 'chapter' in link:
            works_only.remove(link)
        else:
            continue
    return works_only

In [11]:
def save_links_to_txt(file_path, links_to_scrape):
    """Writes the list of fanwork links from works_only_final to a .txt file."""

    # Using "with open" syntax to automatically close the file
    with open(file_path, 'w') as file:
        # Join the list elements into a single string with a newline character
        data_to_write = '\n'.join(links_to_scrape)

        # Write the data to the file
        file.write(data_to_write)

    print(f"The list of links to scrape has been written to {file_path}.")

Now that I know this process works, I can get fanwork links from all the pagination links in "links_to_scrape."

# Testing - One Pagination Link

In [12]:
# test get_links_to_scrape
links_to_scrape = get_links_to_scrape('../data/pagination_links.txt')

['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=5\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=6\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=7\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=8\n', 'https://archiveofourown.org/tags/Cande

In [16]:
# test get_all_links with one pagination link
pagination_link1 = links_to_scrape[0]
print(pagination_link1)

https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

<class 'str'>


In [74]:
# get BeautifulSoup object
soup = get_html_doc_pagination(pagination_link1)

In [75]:
# get work links
all_links = get_all_links(soup)
urls_only = get_urls_only(all_links)
work_links = get_work_links(urls_only)
work_links_pared = pare_down_work_links(work_links)
print(len(work_links_pared))

29


In [76]:
# remove chapter links
work_links_nochap = remove_chapter_links(work_links_pared)
print(len(work_links_nochap))

20


In [77]:
# save fanwork links to .txt file
filepath = ('../data/txt_files/candela_works_page1.txt')
save_links_to_txt(filepath, work_links_nochap)

The list of links to scrape has been written to ../data/txt_files/candela_works_page1.txt.


# Testing - List of Pagination Links
Now that I've successfully saved the first page of fanwork links to a .txt file, I'll do the same with the rest of the links in links_to_scrape.

In [132]:
# get list of Candela Obscura pagination links 2-11
remaining_links_to_scrape = links_to_scrape[1:]
print("There are", str(len(remaining_links_to_scrape)), "remaining links to scrape.")

There are 10 remaining links to scrape.


In [127]:
# iterate through the list following the same steps as the first test
for i in range(len(remaining_links_to_scrape)):
    current_link = remaining_links_to_scrape[i]
    print("Current link: ", str(i+2), " of ", str(len(remaining_links_to_scrape)+1))

    # get BeautifulSoup object
    soup = get_html_doc_pagination(current_link)

    # get work links
    all_links = get_all_links(soup)
    urls_only = get_urls_only(all_links)
    work_links = get_work_links(urls_only)
    work_links_pared = pare_down_work_links(work_links)

    # remove chapter links
    work_links_nochap = remove_chapter_links(work_links_pared)
    print("Length of work_links_nochap: ", str(len(work_links_nochap)))

    # save fanwork links to .txt file
    filepath = ('../data/txt_files/candela_works_page' + str(i+2) + '.txt')
    save_links_to_txt(filepath, work_links_pared)

Current link:  2  of  11
TOS agreement section is present:  True
Length of work_links_nochap:  20
The list of links to scrape has been written to ../data/txt_files/candela_works_page2.txt.
Current link:  3  of  11
TOS agreement section is present:  True
Length of work_links_nochap:  20
The list of links to scrape has been written to ../data/txt_files/candela_works_page3.txt.
Current link:  4  of  11
TOS agreement section is present:  True
Length of work_links_nochap:  20
The list of links to scrape has been written to ../data/txt_files/candela_works_page4.txt.
Current link:  5  of  11
TOS agreement section is present:  True
Length of work_links_nochap:  20
The list of links to scrape has been written to ../data/txt_files/candela_works_page5.txt.
Current link:  6  of  11
TOS agreement section is present:  True
Length of work_links_nochap:  20
The list of links to scrape has been written to ../data/txt_files/candela_works_page6.txt.
Current link:  7  of  11
TOS agreement section is prese

# Next Steps
Now that I have a set of text files with lists of fanwork links, I can feed each fanwork link to the functions in parse_html_docs.ipynb and pull the metadata from each fanwork into a JSON array.